# CSE498R baseline — Step 7: Tier 0 runs on ARC-Easy

Runtime: **GPU, T4**. Arms v46/A3, F0, P0, P1 × seeds 0, 1, 2 on the full
ARC-Easy splits (2251 train / 570 validation / 2376 test), PD-7 / PD-10
protocol (at most 20 epochs, early stopping on validation accuracy with
patience 5, deterministic), the canonical Tier 0 cache. Then the LLM option
likelihood, LLM letter pick (B4) and evidence-weighted vote (B5) baselines on
test, and `evaluation/stats.py` with the PD-14 families.

Everything is written to `/content/drive/MyDrive/qAIR_CSE498R/step7_<commit>/`
(`ckpt/`, `baselines/`); logs go to `runs/<UTC time>_<commit>_step7s<N>/`.
The cache in `cache/arc_easy/` is only read, and its hashes are checked
against the cache-build manifest before anything trains.

| Session | Trains / runs | Cells |
|---|---|---|
| 1 | **F0** (primary arm, PD-11), seeds 0, 1, 2 | 0–6 |
| 2 | A3 (v46), seeds 0, 1, 2 | 0–6 |
| 3 | P0, seeds 0, 1, 2 | 0–6 |
| 4 | P1, seeds 0, 1, 2 | 0–6 |
| 5 | baselines on test, then the statistics | 0–5, 7, 8 |

**Each session:** fresh T4 runtime; in cell 0 set `SESSION_NUMBER`; keep
`PINNED_COMMIT` the same in all five sessions. Cell 4 shows which runs are
finished and which session is next.

**After a disconnect:** connect a fresh runtime, run cells 0–6 again with
the same `SESSION_NUMBER`. Finished runs are skipped; an interrupted run
resumes from its last epoch checkpoint with its RNG state, so it trains the
epochs an uninterrupted run would have trained. If a run refuses to resume
(`CheckpointMismatchError`), stop and paste the message to Claude Code; do not
delete or move checkpoint files.

**Test split:** each run evaluates test once, from the checkpoint with the
best validation accuracy, at the end of training. A finished run is never
started again, so its test split is not evaluated twice. Nothing here is
selected or tuned on test (CLAUDE.md rule 5).

## 0 · Settings and helpers

In [ ]:
# ---- Settings: edit only these ----------------------------------------
PINNED_COMMIT = "REPLACE_WITH_THE_COMMIT_IN_docs/HANDOFF.md"   # the same in all five sessions
SESSION_NUMBER = 1           # 1-4: train F0, A3, P0, P1; 5: baselines and statistics
RUNS_APPROVED = False        # True only after the Step 7 time estimate has been approved
GEN_SEED = 0                 # the fixed generation seed, config.GEN_SEED (checked in cell 1)
ARC_CONFIG = "ARC-Easy"
# ------------------------------------------------------------------------

REPO_URL = "https://github.com/Thorfast191/CSE498R-qAIR.git"
REPO = "/content/CSE498R-qAIR"                  # code, on the local disk
ROOT = "/content/drive/MyDrive/qAIR_CSE498R"    # everything this notebook writes
CACHE = f"{ROOT}/cache/arc_easy"
CKPT = f"{ROOT}/ckpt"
RUNS = f"{ROOT}/runs"
EXPORTS = f"{ROOT}/exports"
# The canonical Tier 0 cache-build manifest (commit 44fc12a), on Drive.
CANONICAL_MANIFEST = f"{RUNS}/20261028_135552_44fc12a/cache_manifest.json"

import hashlib
import json
import os
import re
import subprocess
import sys
import time

# Never read or write an older project folder (qAIR_V45, qAIR_V44, ...).
for path in (ROOT, CACHE, CKPT, RUNS, EXPORTS):
    if re.search(r"qair_v\d", path, re.IGNORECASE):
        raise RuntimeError(f"STOP: {path} names an old project folder.")

if not re.fullmatch(r"[0-9a-f]{7,40}", PINNED_COMMIT):
    raise RuntimeError("STOP: set PINNED_COMMIT to the commit given in docs/HANDOFF.md.")
if SESSION_NUMBER not in (1, 2, 3, 4, 5):
    raise RuntimeError("STOP: SESSION_NUMBER is 1, 2, 3, 4 or 5.")

# One folder for all five sessions of this commit: runs resume from it and
# the statistics read every result in it.
STEP7 = f"{ROOT}/step7_{PINNED_COMMIT[:7]}"
STEP7_CKPT = f"{STEP7}/ckpt"
STEP7_BASELINES = f"{STEP7}/baselines"


def run(cmd, log=None, check=True):
    """
    Runs cmd in the repo, streams its output live (progress bars included)
    and appends it to log.
    """
    env = {**os.environ, "PYTHONUNBUFFERED": "1"}
    proc = subprocess.Popen([str(c) for c in cmd], env=env,
                            cwd=REPO if os.path.isdir(REPO) else None,
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    sink = open(log, "a") if log else None
    pending = b""

    def emit(segment):
        text = segment.decode("utf-8", errors="replace")
        if sink:
            sink.write(text)
            sink.flush()
        sys.stdout.write(text)
        sys.stdout.flush()

    while True:
        chunk = proc.stdout.read1(1 << 16)
        if not chunk:
            break
        pending += chunk
        while True:
            cuts = [i for i in (pending.find(b"\n"), pending.find(b"\r")) if i >= 0]
            if not cuts:
                break
            emit(pending[:min(cuts) + 1])
            pending = pending[min(cuts) + 1:]
    if pending:
        emit(pending)

    code = proc.wait()
    if sink:
        sink.close()
    if check and code != 0:
        raise RuntimeError(f"STOP: {' '.join(map(str, cmd))} exited with {code}"
                           + (f"; full output in {log}" if log else ""))
    return code


def output(cmd):
    """stdout of cmd run in the repo."""
    return subprocess.run([str(c) for c in cmd], cwd=REPO, capture_output=True,
                          text=True, check=True).stdout.strip()


def sha256_text(text):
    return hashlib.sha256(text.encode()).hexdigest()


print("settings ok; PINNED_COMMIT =", PINNED_COMMIT, "; SESSION_NUMBER =", SESSION_NUMBER,
      "; RUNS_APPROVED =", RUNS_APPROVED)

## 1 · Drive and code at the pinned commit

In [ ]:
from google.colab import drive

drive.mount("/content/drive")
for path in (CACHE, CANONICAL_MANIFEST):
    if not os.path.exists(path):
        raise RuntimeError(f"STOP: {path} does not exist; the Step 6 cache build wrote it.")
for path in (RUNS, STEP7_CKPT, STEP7_BASELINES):
    os.makedirs(path, exist_ok=True)

if not os.path.isdir(REPO):
    run(["git", "clone", "--quiet", REPO_URL, REPO])
run(["git", "fetch", "--quiet", "origin"])
run(["git", "checkout", "--quiet", "--detach", PINNED_COMMIT])

COMMIT = output(["git", "rev-parse", "HEAD"])
if not COMMIT.startswith(PINNED_COMMIT):
    raise RuntimeError(f"STOP: HEAD is {COMMIT}, not {PINNED_COMMIT}.")
if output(["git", "status", "--porcelain", "--untracked-files=no"]):
    raise RuntimeError("STOP: the clone has uncommitted changes; every run would be marked dirty.")

config_seed = int(output([sys.executable, "-c", "import config; print(config.GEN_SEED)"]))
if config_seed != GEN_SEED:
    raise RuntimeError(f"STOP: GEN_SEED is {GEN_SEED} here but config.GEN_SEED is {config_seed}.")

SESSION = (f"{RUNS}/{time.strftime('%Y%m%d_%H%M%S', time.gmtime())}_{COMMIT[:7]}"
           f"_step7s{SESSION_NUMBER}")
os.makedirs(SESSION)

print("commit :", COMMIT)
print("session:", SESSION)
print("step 7 :", STEP7)

## 2 · Environment before installing anything (E11)

In [ ]:
ENV_REPORT = """
import torch
print("torch             ", torch.__version__)
print("torch.version.cuda", torch.version.cuda)
print("cuda available    ", torch.cuda.is_available())
print("gpu               ", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")
"""
run([sys.executable, "-c", ENV_REPORT], log=f"{SESSION}/env_before.txt")

freeze_before = output([sys.executable, "-m", "pip", "freeze"])
with open(f"{SESSION}/env_before.txt", "a") as f:
    f.write("\n# pip freeze\n" + freeze_before + "\n")
print("\n# pip freeze (before install)\n" + freeze_before)

TORCH = output([sys.executable, "-c", "import torch; print(torch.__version__)"])
CUDA = output([sys.executable, "-c", "import torch; print(torch.version.cuda)"])
GPU = output([sys.executable, "-c", "import torch; print(torch.cuda.get_device_name(0) "
              "if torch.cuda.is_available() else 'none')"])

if GPU == "none":
    raise RuntimeError("STOP: no GPU. Runtime -> Change runtime type -> T4 GPU.")
if "T4" not in GPU:
    print(f"\n[WARNING] GPU is {GPU}, not a T4; timings will not match the T4 plan.")

## 3 · Clean up, install the pins, check

1. Uninstalls the packages that conflicted in the v46 run (diffusers, gradio,
   jax, jaxlib) and jax's dependents (flax, optax, chex, orbax-checkpoint).
2. Installs `requirements.txt` with torch held at Colab's build (never
   reinstalled). This downgrades transformers, huggingface_hub, accelerate,
   datasets and their dependencies to the pins.
3. Tries `pennylane-lightning[gpu]` (built for CUDA 12; Colab runs CUDA 13).
   If it fails, the circuit runs on `lightning.qubit` and the manifest says so.
4. Classifies every `pip check` complaint: a package qAIR needs (directly or
   transitively) stops the notebook; a Colab kernel package is kept and
   documented; anything else is unused by qAIR and uninstalled. Every step
   is in `install.log`.

Expected from the 2026-10-28 freeze [DERIVED, simulated from PyPI metadata]:
step 4 removes gcsfs, gradio-client, then bigframes and hf-gradio, plus
dopamine-rl (needs flax); ipython keeps its pre-existing missing `jedi`.

In [ ]:
INSTALL_LOG = f"{SESSION}/install.log"


def note(text):
    """Printed and appended to install.log."""
    print(text)
    with open(INSTALL_LOG, "a") as f:
        f.write(f"\n# {text}\n")


def normalize(name):
    return re.sub(r"[-_.]+", "-", name).lower()


def qair_imports():
    """Top-level modules imported by any tracked .py file of the repo."""
    names = set()
    for path in output(["git", "ls-files", "*.py"]).split():
        with open(f"{REPO}/{path}", encoding="utf-8") as f:
            for line in f:
                match = re.match(r"\s*(?:import|from)\s+([A-Za-z_]\w*)", line)
                if match:
                    names.add(normalize(match.group(1)))
    return names


CLOSURE = """
import importlib.metadata as md, json, re, sys
from packaging.requirements import Requirement
norm = lambda n: re.sub(r"[-_.]+", "-", n).lower()
seen, todo = set(), [norm(n) for n in sys.argv[1:]]
while todo:
    name = todo.pop()
    if name in seen:
        continue
    try:
        requires = md.requires(name) or []
    except md.PackageNotFoundError:
        continue
    seen.add(name)
    for spec in requires:
        req = Requirement(spec)
        if req.marker is None or req.marker.evaluate({"extra": ""}):
            todo.append(norm(req.name))
print(json.dumps(sorted(seen)))
"""


def qair_needs():
    """qAIR's packages and everything they require, transitively."""
    runtime = output([sys.executable, "-c", "from training.environment import RUNTIME_PACKAGES; "
                      "print(' '.join(RUNTIME_PACKAGES))"]).split()
    roots = runtime + ["numpy", "tqdm", "matplotlib", "pandas", "scikit-learn"]
    return set(json.loads(output([sys.executable, "-c", CLOSURE, *roots])))


def pip_check_complaints():
    """{package: [pip check lines]} for every package pip check complains about."""
    report = subprocess.run([sys.executable, "-m", "pip", "check"], capture_output=True, text=True)
    found = {}
    for line in (report.stdout + report.stderr).splitlines():
        match = re.match(r"(\S+) \S+ (?:requires|has requirement|is not supported)", line)
        if match:
            found.setdefault(normalize(match.group(1)), []).append(line.strip())
    return found


# Colab's own kernel and UI; a complaint about one of these is documented, never acted on.
COLAB_KEEP = {"google-colab", "ipython", "ipykernel", "jupyter-client", "jupyter-core",
              "jupyter-server", "notebook", "ipywidgets"}

# 1. Packages that conflicted in the v46 run and jax's dependents.
QAIR_IMPORTS = qair_imports()
FIRST = ["diffusers", "gradio", "jax", "jaxlib", "flax", "optax", "chex", "orbax-checkpoint"]
if QAIR_IMPORTS & {normalize(p).replace("-", "_") for p in FIRST} | QAIR_IMPORTS & set(FIRST):
    raise RuntimeError("STOP: qAIR imports a package on the uninstall list. Tell Claude Code.")
note("uninstalling (v46 conflicts, jax family): " + " ".join(FIRST))
run([sys.executable, "-m", "pip", "uninstall", "-y", *FIRST], log=INSTALL_LOG)

# 2. The pins, torch held at Colab's build.
with open(f"{REPO}/requirements.txt") as f:
    requirements = f.read().splitlines()

torch_line = re.compile(r"\s*torch\s*(==|$)")
torch_pin = next((line.split("==", 1)[1].strip() for line in requirements
                  if torch_line.match(line) and "==" in line), None)
if torch_pin is not None and torch_pin != TORCH:
    raise RuntimeError(f"STOP: requirements.txt pins torch=={torch_pin}, Colab ships {TORCH}. "
                       f"Do not reinstall torch; tell Claude Code.")

without_torch = f"{SESSION}/requirements_without_torch.txt"
with open(without_torch, "w") as f:
    f.write("\n".join(line for line in requirements if not torch_line.match(line)) + "\n")
keep_torch = f"{SESSION}/keep_torch.txt"
with open(keep_torch, "w") as f:
    f.write(f"torch=={TORCH}\n")

run([sys.executable, "-m", "pip", "install", "-q", "-r", without_torch, "-c", keep_torch],
    log=INSTALL_LOG)

# 3. Optional GPU simulator. The 0.42.0 build targets CUDA 12 and Colab runs CUDA 13:
#    if it does not install, the quantum layer runs on lightning.qubit (recorded in the manifest).
code = run([sys.executable, "-m", "pip", "install", "-q", "-c", keep_torch,
            "pennylane-lightning[gpu]==0.42.0"], log=INSTALL_LOG, check=False)
LIGHTNING_GPU_INSTALLED = code == 0
note("pennylane-lightning[gpu]==0.42.0 " + ("installed" if LIGHTNING_GPU_INSTALLED else
     f"did NOT install (exit {code}); the circuit will run on lightning.qubit, ~20-30x slower"))

torch_after = output([sys.executable, "-c", "import torch; print(torch.__version__)"])
if torch_after != TORCH:
    raise RuntimeError(f"STOP: torch changed from {TORCH} to {torch_after}. "
                       f"Runtime -> Disconnect and delete runtime, then tell Claude Code.")

# 4. Every pip check complaint, classified.
NEEDED = qair_needs()
removed, kept = set(FIRST), {}
for _ in range(6):
    complaints = pip_check_complaints()
    needed = {p: lines for p, lines in complaints.items()
              if p in NEEDED or p.replace("-", "_") in QAIR_IMPORTS}
    if needed:
        raise RuntimeError("STOP: pip check complains about packages qAIR needs (must fix; tell "
                           "Claude Code):\n" + "\n".join(l for ls in needed.values() for l in ls))
    kept = {p: lines for p, lines in complaints.items() if p in COLAB_KEEP}
    unused = sorted(set(complaints) - set(kept))
    if not unused:
        break
    for p in unused:
        note(f"unused by qAIR, uninstalling {p}: " + " | ".join(complaints[p]))
    run([sys.executable, "-m", "pip", "uninstall", "-y", *unused], log=INSTALL_LOG)
    removed |= set(unused)
for p, lines in kept.items():
    note(f"Colab kernel package, kept (harmless for qAIR) {p}: " + " | ".join(lines))
note("removed in total: " + " ".join(sorted(removed)))

pip_check = subprocess.run([sys.executable, "-m", "pip", "check"], capture_output=True, text=True)
PIP_CHECK_OK = pip_check.returncode == 0
freeze_after = output([sys.executable, "-m", "pip", "freeze"])
with open(f"{SESSION}/env_after.txt", "w") as f:
    f.write("# pip check\n" + pip_check.stdout + pip_check.stderr
            + "\n# pip freeze\n" + freeze_after + "\n")

FREEZE_SHA = sha256_text(freeze_after)
PACKAGES_SHA = output([sys.executable, "-c", "from training.environment import "
                       "environment_record; print(environment_record()[1]['packages_sha256'])"])

print("\n# pip check\n" + (pip_check.stdout + pip_check.stderr).strip())
print("\n" + "=" * 66)
print("ENVIRONMENT (paste to Claude Code)")
print("=" * 66)
print("commit            ", COMMIT)
print("torch             ", TORCH)
print("torch.version.cuda", CUDA)
print("gpu               ", GPU)
print("lightning.gpu     ", "installed" if LIGHTNING_GPU_INSTALLED else "NOT installed")
print("pip check ok      ", PIP_CHECK_OK, f"(kept, documented: {sorted(kept)})" if kept else "")
print("removed           ", " ".join(sorted(removed)))
print("pip freeze sha256 ", FREEZE_SHA)
print("packages_sha256   ", PACKAGES_SHA, " (training/environment.py; the manifests record this one)")
print("full lists        ", f"{SESSION}/env_before.txt, {SESSION}/env_after.txt, {INSTALL_LOG}")
if torch_pin is None:
    print("\nSTOP 1: torch is not pinned yet. Paste the block above to Claude Code; it commits "
          "torch==<version> and gives you the new PINNED_COMMIT.")
else:
    print(f"\ntorch pinned ({torch_pin}) and installed; continue.")

## 4 · Status: which runs are finished, where to continue

Safe to run at any time; reads only. `epochs (bN)`: epochs trained, and the
epoch whose checkpoint the test number was read from. `in progress` resumes
in its session; `trained, test pending` evaluates test when its session runs
again.

In [ ]:
STATUS = [sys.executable, "scripts/step7_runs.py", "status",
          "--ckpt-dir", STEP7_CKPT, "--baselines-dir", STEP7_BASELINES]
run(STATUS, log=f"{SESSION}/status.txt")

## 5 · Canonical cache check

Hashes the three cache files and compares them with the cache-build
manifest and with the canonical prefixes (train `4bc0031289a03e4f`,
validation `0760535c445ad6dc`, test `5faa6d62c82612d6`). Cells 6 and 7 check
again before they start.

In [ ]:
run([sys.executable, "scripts/step7_runs.py", "verify-cache",
     "--cache-dir", CACHE, "--manifest", CANONICAL_MANIFEST], log=f"{SESSION}/cache_check.log")

## 6 · Train this session's arm (sessions 1–4)

Three runs, seeds 0, 1, 2, one after another, each `main.py --mode ablation`
with the defaults (20 epochs, patience 5, deterministic, torch circuit
engine). Up to ~3.7 h on a T4 (estimate in `docs/HANDOFF.md`); keep the tab
open. Every epoch writes a checkpoint to Drive.

In [ ]:
if not RUNS_APPROVED:
    raise RuntimeError("STOP: set RUNS_APPROVED = True in cell 0 only after the Step 7 "
                       "time estimate has been approved.")

if SESSION_NUMBER == 5:
    print("Session 5 trains nothing: run cells 7 and 8.")
else:
    try:
        run([sys.executable, "scripts/step7_runs.py", "train", "--session", SESSION_NUMBER,
             "--cache-dir", CACHE, "--ckpt-dir", STEP7_CKPT, "--manifest", CANONICAL_MANIFEST],
            log=f"{SESSION}/train_session{SESSION_NUMBER}.log")
    except RuntimeError:
        print("\nIf the output above ends in CheckpointMismatchError or a strict-determinism "
              "error: STOP, paste it to Claude Code, and leave the checkpoint files where they are.")
        raise
    finally:
        run(STATUS, log=f"{SESSION}/status.txt", check=False)

## 7 · Baselines on test (session 5)

LLM option likelihood and the B5 evidence-weighted vote read the cache; the
B4 letter pick loads Qwen2.5-0.5B-Instruct and refuses if its revision is
not the cache's. Each writes `baselines/<mode>_test.json` once; a baseline
whose file exists is not run again.

In [ ]:
if not RUNS_APPROVED:
    raise RuntimeError("STOP: set RUNS_APPROVED = True in cell 0 only after approval.")
if SESSION_NUMBER != 5:
    raise RuntimeError(f"STOP: this is session {SESSION_NUMBER}; the baselines run in session 5.")

run([sys.executable, "scripts/step7_runs.py", "baselines", "--cache-dir", CACHE,
     "--out-dir", STEP7_BASELINES, "--manifest", CANONICAL_MANIFEST],
    log=f"{SESSION}/baselines.log")

## 8 · Results table (session 5, after all 12 runs and 3 baselines)

1. Rebuilds `ckpt/ablation_results.pt` from the 12 result files; it refuses
   if they come from different code or caches.
2. Per arm: validation and test accuracy (mean ± std over the seeds) and the
   collapse diagnostics.
3. The three baselines' test accuracy. The B5 vote is pre-registered and
   reported; it is in neither PD-14 family.
4. `evaluation/stats.py`: the PRIMARY family (F0 vs LLM option likelihood
   and letter pick, Holm over 2, the PD-11 verdict), the SECONDARY family
   (A3, P0, P1 × the two LLM baselines, Holm over 6, reported only), and an
   exploratory, uncorrected table against A3.

Everything printed is also in `results_table.txt` in this session's folder.
Paste all of it to Claude Code.

In [ ]:
TABLE = f"{SESSION}/results_table.txt"

DONE = """
import sys
sys.path.insert(0, "scripts")
import step7_runs
_, _, next_session = step7_runs.status(sys.argv[1], sys.argv[2])
sys.exit(0 if next_session is None else 1)
"""
if run([sys.executable, "-c", DONE, STEP7_CKPT, STEP7_BASELINES], check=False):
    run(STATUS)
    raise RuntimeError("STOP: not every run and baseline is finished (status above).")

SUMMARY = """
import json, os, sys
import torch
from training.ablations import summarize
from training.results import COMBINED_NAME, combine_results, mixing_note

ckpt, baselines = sys.argv[1], sys.argv[2]
combined = combine_results(ckpt)
torch.save(combined, os.path.join(ckpt, COMBINED_NAME))
summarize(combined["results"])
print(mixing_note(combined))
print()
for mode, label in (("llm", "LLM option likelihood"), ("letter", "LLM letter pick (B4)"),
                    ("vote", "evidence-weighted vote (B5, pre-registered; no PD-14 family)")):
    with open(os.path.join(baselines, f"{mode}_test.json")) as f:
        saved = json.load(f)
    hits = sum(saved["records"])
    print(f"[BASELINE test] {label:<62s} {saved['accuracy']:.4f}  ({hits}/{saved['n']})")
"""

with open(TABLE, "w") as f:
    f.write(f"RESULTS TABLE  commit {COMMIT}  session {SESSION}\n")
print(f"RESULTS TABLE  commit {COMMIT}  session {SESSION}")
run([sys.executable, "scripts/step7_runs.py", "verify-cache", "--cache-dir", CACHE,
     "--manifest", CANONICAL_MANIFEST], log=TABLE)
run([sys.executable, "-c", SUMMARY, STEP7_CKPT, STEP7_BASELINES], log=TABLE)
run([sys.executable, "-m", "evaluation.stats", "--results", f"{STEP7_CKPT}/ablation_results.pt",
     "--baselines-dir", STEP7_BASELINES, "--baseline", "A3_persistent"], log=TABLE)
print(f"\nPaste everything from RESULTS TABLE down to Claude Code (also in {TABLE}).")